In [1]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import psycopg2
from pgvector.psycopg2 import register_vector
import gradio as gr
from sklearn.metrics import ndcg_score
import numpy as np
import os
import json
import uuid

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT).to(device)

model.classifier[2] = torch.nn.Identity()
model.eval()

# 3. Transformaciones específicas (ConvNeXt usa 224x224 y normalización estándar)
transform = transforms.Compose([
    transforms.Resize(236), # Un poco más grande para el crop
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def get_embedding(image_path_or_pil):
    if isinstance(image_path_or_pil, str):
        img = Image.open(image_path_or_pil).convert('RGB')
    else:
        img = image_path_or_pil.convert('RGB')
        
    img_t = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        embedding = model(img_t).cpu().numpy().flatten()
    return embedding.tolist()

Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /home/tobias/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:02<00:00, 56.7MB/s] 


In [3]:
conn = psycopg2.connect(
    dbname="dogs", 
    user="dogs_user", 
    password="dogs_pass", 
    host="localhost", 
    port="5432"
)
cur = conn.cursor()
cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
register_vector(conn)

cur.execute("""
    CREATE TABLE IF NOT EXISTS dog_embeddings (
        id_imagen VARCHAR PRIMARY KEY,
        embedding vector(768),
        path VARCHAR NOT NULL,
        breed VARCHAR NOT NULL,
        metadata JSONB
    );
""")
conn.commit()

In [5]:
def indexar_dataset(dataset_path):
    for breed in os.listdir(dataset_path):
        breed_path = os.path.join(dataset_path, breed)
        if not os.path.isdir(breed_path): continue
            
        for img_name in os.listdir(breed_path):
            img_path = os.path.join(breed_path, img_name)
            emb = get_embedding(img_path)
            img_id = str(uuid.uuid4())
            
            cur.execute("""
                INSERT INTO dog_embeddings (id_imagen, embedding, path, breed, metadata)
                VALUES (%s, %s, %s, %s, %s)
            """, (img_id, emb, img_path, breed, json.dumps({"source": "train"})))
    conn.commit()

indexar_dataset("data/dataset/train")

In [6]:
def search_similar(query_embedding, top_k=10, metric='cosine'):
    if metric == 'cosine':
        query = """
            SELECT path, breed, 1 - (embedding <=> %s::vector) AS score
            FROM dog_embeddings ORDER BY embedding <=> %s::vector LIMIT %s;
        """
    else:
        query = """
            SELECT path, breed, (embedding <-> %s::vector) AS score
            FROM dog_embeddings ORDER BY embedding <-> %s::vector LIMIT %s;
        """
        
    cur.execute(query, (query_embedding, query_embedding, top_k))
    return cur.fetchall()

In [ ]:
def predict_gradio(image):
    emb = get_embedding(image)
    results = search_similar(emb, top_k=10, metric='cosine')
    
    similar_images = [(res[0], res[1]) for res in results]
    predicted_breed = results[0][1] if results else "Desconocida"
    
    return image, similar_images, predicted_breed

interface = gr.Interface(
    fn=predict_gradio,
    inputs=gr.Image(type="pil"),
    outputs=[
        gr.Image(label="Imagen Consultada"),
        gr.Gallery(label="Top 10 Similares"),
        gr.Textbox(label="Raza Predicha")
    ],
    title="Buscador de Razas por Similitud"
)

interface.launch()